# Using MCP Tools with LangChain

In this notebook, we will explore how to create a reasoning action agent using tools exposed by an MCP server with LangChain.

## Recommended Hardware

This notebook can run on the following hardware or remote resources.

✅ AMD Instinct™ Accelerators  
✅ AMD Radeon™ RX/PRO Graphics Cards  
✅ AMD EPYC™ Processors  
✅ AMD Ryzen™ (AI) Processors  

[![Open in AMD Developer Cloud](https://img.shields.io/badge/Open_in_AMD_Developer_Cloud-000000?logo=amd&logoSize=auto)](https://amd-ai-academy.com/github/AMDResearch/aup-ai-tutorials/blob/main/ai-agents/02-a-mcp-tools.ipynb)  

## Software Environment

Install ROCm on your system.

| Linux | Windows |
|-------|---------|
| [Install PyTorch](https://rocm.docs.amd.com/projects/install-on-linux/en/latest/install/quick-start.html) | [PyTorch on Windows](https://rocm.docs.amd.com/projects/radeon-ryzen/en/latest/docs/install/installrad/windows/install-pytorch.html)|
| [Install Docker container](https://amdresearch.github.io/aup-ai-tutorials//env/env-gpu.html) | |

## Goals

- Launch an MCP server and connect to it from a Python client.
- Create a ReAct agent using LangChain with MCP tools.
- Understand how tool binding, tool calling, and result passing work step by step.

### Install Dependencies

Install the package dependencies needed for this notebook or series of notebooks.

First, get the `aup_config.py` script locally if needed. Then install the dependencies (`aup_setup()`). This step may take a few minutes and only needs to be done once.

In [ ]:
![ -f aup_config.py ] || wget https://raw.githubusercontent.com/AMDResearch/aup-ai-tutorials/refs/heads/main/ai-agents/aup_config.py

In [ ]:
from aup_config import aup_setup
aup_setup()

## Enhance the LLM with Tools

Use `ChatOpenAI` to connect to the Ollama local endpoint using its OpenAI-compatible API

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.tools import load_mcp_tools
from langchain.agents import create_agent
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_core.messages import HumanMessage, ToolMessage

In [ ]:
model = ChatOpenAI(model="qwen3.5:9b", base_url="http://localhost:11434/v1/", api_key="ollama")

### Launch MCP Server

Using `StdioServerParameters`, we can launch an MCP server in the background and communicate with it using `stdio`. You can check the Python file [math_server.py](math_server.py), which uses `FastMCP` to expose functions as tools.

In [ ]:
server_params = StdioServerParameters(command="python", args=["math_server.py"])

### Create stdio Session

Once the MCP server is running, we can start the `stdio_client` session. The cell below queries the available MCP tools and displays them.

In [ ]:
async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:

        await session.initialize()

        # Convert MCP tools to LangChain tools
        tools = await load_mcp_tools(session)
        print(f"Available tools:")
        for tool in tools:
            print(f'{tool}\n')

### Create Async ReAct Agent

We can now create a ReAct (Reasoning and Acting) agent using `create_agent` from LangGraph. This function takes the model and the available tools as parameters. The agent will first reason about the query and then use the available tools if needed.

In [ ]:
async def run_react_agent(query: str):
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await load_mcp_tools(session)
            agent = create_agent(model, tools)
            agent_response = await agent.ainvoke({"messages": query})
            return agent_response

Now we can query the ReAct agent. Note that in the response the `square_root` tool is called and the result is used as part of the answer.

In [ ]:
query = "what is the square root of 55?"
agent_response = await run_react_agent(query)
for m in agent_response['messages']:
    m.pretty_print()

Similarly, if we ask for a multiplication operation, the multiplication tool is called.

In [ ]:
query = "what is the result of 3.14 * 5?"
agent_response = await run_react_agent(query)
for m in agent_response['messages']:
    m.pretty_print()

Again, the division tool is called

In [ ]:
query = "what is the result of 99 / 2.3?"
agent_response = await run_react_agent(query)
for m in agent_response['messages']:
    m.pretty_print()

Note that the agent does not answer when it cannot find a suitable tool for the query

In [ ]:
query = "what is the capital of Ireland?"
agent_response = await run_react_agent(query)
for m in agent_response['messages']:
    m.pretty_print()

## Understanding how LLMs Work with Tools

In the previous section, `create_agent` handled all the details of binding tools, invoking the LLM, and executing tool calls. We will now break that process down step by step to understand what happens under the hood.

### Bind Tools to the LLM

The first step is to inform the LLM of the available tools. When we call `model.bind_tools(tools)`, the tool schemas (name, description, and parameter types) are converted into the OpenAI function-calling format and attached as a `tools` parameter in every API request. This is **not** injected into the system prompt, it is sent as a separate structured field alongside the `messages`. The LLM itself **does not execute** any tools, it only decides which tool to call and with what arguments.

In [ ]:
async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools = await load_mcp_tools(session)

        model_with_tools = model.bind_tools(tools)

        # Inspect the tool definitions that are sent to the LLM with every API request
        bound_tools = model_with_tools.kwargs['tools']
        for t in bound_tools:
            func = t['function']
            print(f"Tool: {func['name']}")
            print(f"  Description: {func['description']}")
            print(f"  Parameters: {func['parameters']}\n")

### The LLM Decides Which Tool to Call

When we invoke the model with a query, the LLM analyzes the query and the available tool schemas. Instead of answering directly, it returns an `AIMessage` containing `tool_calls`, a structured request indicating which tool to invoke and with what arguments.

In [ ]:
async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools = await load_mcp_tools(session)
        model_with_tools = model.bind_tools(tools)

        messages = [HumanMessage(content="what is the square root of 55?")]
        ai_response = await model_with_tools.ainvoke(messages)

        print("Content:", ai_response.content)
        print("\nTool calls requested by the LLM:")
        for tc in ai_response.tool_calls:
            print(f"  Function: {tc['name']}, Arguments: {tc['args']}")

### Execute the Tool and Pass the Result Back

The LLM does not run the tool itself, the **client** is responsible for executing it. We use `session.call_tool()` to invoke the MCP tool by name with the arguments the LLM chose. The result is then wrapped in a `ToolMessage` and appended to the conversation history. Finally, we invoke the LLM again with the full message list (including the tool results), and the LLM uses them to produce its final answer.

In [ ]:
async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools = await load_mcp_tools(session)
        model_with_tools = model.bind_tools(tools)

        messages = [HumanMessage(content="what is the square root of 55?")]
        ai_response = await model_with_tools.ainvoke(messages)
        messages.append(ai_response)

        for tool_call in ai_response.tool_calls:
            result = await session.call_tool(tool_call['name'], arguments=tool_call['args'])
            print(f"Tool executed: {tool_call['name']}({tool_call['args']}) → {result}")
            # Append the tool result as a ToolMessage
            messages.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))

        # Send the full conversation (including tool results) back to the LLM
        print(f"Message sent to the LLM:")
        for message in messages:
            print(f"{message}\n")
        final_response = await model_with_tools.ainvoke(messages)
        print(f"\nFinal LLM answer: {final_response.content}")

## Exercises for the Reader

- Add your own MCP tool to `math_server.py` (for example, a `power` tool) and test it with the agent.
- Make one of the MCP tools return an incorrect value. What happens?

## Conclusions

In this notebook we launched an MCP server using `FastMCP` and connected to it via `stdio`. We then loaded the MCP tools and used them with a LangChain ReAct agent that reasons about queries and calls the appropriate tool. Finally, we broke down the tool-calling process into its three stages: binding tool schemas to the LLM, receiving structured tool calls from the LLM's response, and executing the tools on the client side before passing results back to the LLM for a final answer.

## References

<div class="alert alert-block alert-info">
<ul>
  <li><a href="https://modelcontextprotocol.io/docs/getting-started/intro">Model Context Protocol</a></li>
</ul>
</div>

---

[AMD University Program](https://www.amd.com/aup).

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.

SPDX-License-Identifier: MIT